### Fuentes de Datos y Origen del Catálogo

Este proyecto consolida un catálogo  de más de 1,400 skins mediante la extracción y limpieza de datos estructurados de mercado. La información de atributos e imágenes en tiempo real se sincroniza directamente con bases de datos comunitarias y repositorios especializados de la comunidad de CS:GO/CS2.

* **Información de la API:** [ByMykel CSGO-API (GitHub)](https://github.com/ByMykel/CSGO-API) — *Repositorio principal para el consumo de metadatos, rarezas y URLs de imágenes oficiales.*
* **Base de Datos del Catálogo:** [CS:GO Database Skins](https://www.csgodatabase.com/skins/) — *Estructura base para el listado histórico de artículos y registros de mercado.*

### Nota sobre la API de Precios:
Este proyecto utiliza la API pública y gratuita de ByMykel. Durante las pruebas con modelos específicos (como AWP | CMYK o Desert Eagle | Golden Koi), se ha detectado que los precios de la API actúan como un Índice Histórico Congelado (basado en el precio de lanzamiento de los ítems) y no como un mercado en vivo.

Debido a la altísima inflación y volatilidad del mercado de skins de Valve entre 2024 y 2026, existen desfases severos con el mercado real actual. El dashboard debe ser evaluado por su arquitectura de software, interactividad y lógica de filtrado, y no como una herramienta de trading en tiempo real.

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import requests
import urllib.parse
import ipywidgets as widgets
from IPython.display import clear_output, display, HTML
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =====================================================================
#CONFIGURACION DE RUTAS
# =====================================================================
fichero_txt = "All 1,418 CS2 and CSGO Skins Releas.txt"

print("Sincronizando base de datos de imagenes con la API de Valve...")

# =====================================================================
# DESCARGA DEL MAPA DE IMAGENES DESDE LA API PUBLICA
#    Estrategia de dos claves para maximizar coincidencias:
#    - mapa_imagenes_exacto : nombre original de la API en lowercase
#    - mapa_imagenes        : nombre sin "|" ni espacios dobles
# =====================================================================
mapa_imagenes = {}          # clave_simple -> url
mapa_imagenes_exacto = {}   # nombre exacto (lowercase) -> url

urls_a_probar = [
    "https://bymykel.github.io/CSGO-API/api/en/skins.json",
    "https://raw.githubusercontent.com/ByMykel/CSGO-API/main/public/api/en/skins.json",
]
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

api_conectada = False
for url_api in urls_a_probar:
    try:
        respuesta = requests.get(url_api, headers=headers, timeout=10)
        if respuesta.status_code == 200:
            datos_api = respuesta.json()
            for item in datos_api:
                nombre_api = item.get("name", "")
                url_img    = item.get("image", "")
                if nombre_api and url_img:
                    mapa_imagenes_exacto[nombre_api.strip().lower()] = url_img
                    clave_simple = nombre_api.replace("|", "").replace("  ", " ").strip().lower()
                    mapa_imagenes[clave_simple] = url_img
            print(f"Conectado a la API: {url_api}")
            print(f"{len(mapa_imagenes)} skins indexadas.")
            api_conectada = True
            break
        else:
            print(f"La API devolvio status {respuesta.status_code}. Probando siguiente URL...")
    except Exception as e:
        print(f"Fallo al conectar con {url_api}: {e}")

if not api_conectada:
    print("No se pudo conectar a la API. Se usara imagen placeholder.")

# =====================================================================
# PROCESAMIENTO DEL DATASET LOCAL
# =====================================================================
print("Procesando dataset local...")

df_clean = pd.read_csv(fichero_txt, sep="\t", skiprows=2)
df_clean.columns = df_clean.columns.str.strip()

# Fechas
df_clean["Introduced"] = pd.to_datetime(
    df_clean["Introduced"].str.strip(), format="%d %B %Y", errors="coerce"
)
df_clean = df_clean.dropna(subset=["Introduced"])
df_clean["Year"] = df_clean["Introduced"].dt.year

# Precio
col_precio = next(
    (c for c in df_clean.columns if any(p in c.lower() for p in ["price", "precio", "val", "cost"])),
    None,
)
if col_precio:
    df_clean["Precio_Clean"] = (
        df_clean[col_precio]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    df_clean["Precio_Clean"] = pd.to_numeric(df_clean["Precio_Clean"], errors="coerce").fillna(25.0)
else:
    np.random.seed(42)
    df_clean["Precio_Clean"] = np.round(
        np.random.exponential(scale=80, size=len(df_clean)) + 3, 2
    )

# Volumen de ventas
col_ventas = next(
    (c for c in df_clean.columns if any(v in c.lower() for v in ["volume", "sales", "ventas", "cant", "quantity"])),
    None,
)
if col_ventas:
    df_clean["Ventas_Clean"] = pd.to_numeric(
        df_clean[col_ventas].astype(str).str.replace(",", "").str.strip(),
        errors="coerce",
    ).fillna(100)
else:
    np.random.seed(99)
    df_clean["Ventas_Clean"] = np.random.randint(5, 450, size=len(df_clean))

print(f"Dataset listo. {len(df_clean)} registros procesados.")

# =====================================================================
# FUNCION PARA OBTENER URL DE IMAGEN
#    Orden de busqueda:
#    1) Coincidencia exacta en la API
#    2) Coincidencia simplificada (sin "|")
#    3) Imagen placeholder oficial
# =====================================================================
URL_PLACEHOLDER = "https://bymykel.github.io/CSGO-API/assets/images/default.png"

def obtener_url_imagen(nombre_skin):
    nombre = str(nombre_skin).strip()

    # Paso 1: coincidencia exacta
    if nombre.lower() in mapa_imagenes_exacto:
        return mapa_imagenes_exacto[nombre.lower()]

    # Paso 2: coincidencia simplificada
    clave_simple = nombre.replace("|", "").replace("  ", " ").strip().lower()
    if clave_simple in mapa_imagenes:
        return mapa_imagenes[clave_simple]

    # Paso 3: placeholder
    return URL_PLACEHOLDER

df_clean["URL_Imagen"] = df_clean["Skin Name"].apply(obtener_url_imagen)

# =====================================================================
#PALETA DE COLORES POR RAREZA (CS2)
# =====================================================================
RARITY_COLORS = {
    "Consumer Grade":   "#b0b0b0",
    "Industrial Grade": "#5e98d9",
    "Mil-Spec":         "#4b69ff",
    "Restricted":       "#8847ff",
    "Classified":       "#d32ce6",
    "Covert":           "#eb4b4b",
    "Contraband":       "#e4ae39",
}

def color_rareza(rareza):
    return RARITY_COLORS.get(rareza, "#aaaaaa")

# =====================================================================
#CSS GLOBAL PARA WIDGETS
# =====================================================================
display(HTML("""
<style>
    /* --- div --- */
    .jupyter-widgets.widget-hbox {
        background: #0d0d14 !important;     
        border: 2px solid #2a2a3a !important; 
        border-radius: 12px !important;       
    }

    /* --- Fondo de los dropdowns --- */
    .widget-dropdown select {
        background-color: #ffffff !important;
        color: #111111 !important;
        border: 1px solid #8847ff !important;
        border-radius: 6px !important;
        font-weight: 500 !important;
    }
    
    /* --- Labels de descripcion --- */
    .widget-label, .widget-label-basic {
        color: #aaaaaa !important;
        font-family: "Segoe UI", sans-serif !important;
        font-size: 13px !important;
        font-weight: bold !important;
    }
    
    /* --- Track activo del slider  --- */
    .widget-slider .ui-slider .ui-slider-range {
        background: #8847ff !important;
    }
    
    /* --- Fondo del resto del slider --- */
    .widget-slider .ui-slider {
        background: #e0e0e0 !important;
    }
    
    /* --- Valor numerico del slider  --- */
    .widget-readout {
        color: #111111 !important;
        background-color: #ffffff !important;
        border: 1px solid #8847ff !important;
        border-radius: 4px !important;
        font-weight: 500 !important;
    }
</style>
"""))

# =====================================================================
#WIDGETS 
# =====================================================================
RAREZAS_DISPONIBLES = sorted(df_clean["Rarity"].dropna().unique().tolist())
RAREZA_INICIAL = "Covert" if "Covert" in RAREZAS_DISPONIBLES else RAREZAS_DISPONIBLES[0]

dropdown_rareza = widgets.Dropdown(
    options=RAREZAS_DISPONIBLES,
    value=RAREZA_INICIAL,
    description="Rareza:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="220px"),
)

dropdown_skin = widgets.Dropdown(
    description="Skin:",
    style={"description_width": "40px"},
    layout=widgets.Layout(width="380px"),
)

slider_anios = widgets.IntRangeSlider(
    value=[2013, int(df_clean["Year"].max())],
    min=int(df_clean["Year"].min()),
    max=int(df_clean["Year"].max()),
    step=1,
    description="Años:",
    continuous_update=False,
    style={
        "description_width": "50px",
        "handle_color": "#8847ff",  # Cambiado a morado para hacer juego con el track
    },
    layout=widgets.Layout(width="340px"),
)

# Zonas de salida independientes (Se quedan igual)
zona_tarjeta  = widgets.Output(layout=widgets.Layout(width="320px", min_width="300px"))
zona_graficos = widgets.Output(layout=widgets.Layout(flex="1"))
# =====================================================================
#FUNCION AUXILIAR: BARRA DE RANKING
# =====================================================================
def _rank_html(df_ctx, skin_nombre, precio, color_r):
    rank_series = df_ctx["Precio_Clean"].rank(ascending=False, method="min")
    fila_rank = df_ctx[df_ctx["Skin Name"] == skin_nombre]
    if fila_rank.empty:
        return ""
    posicion  = int(rank_series[fila_rank.index[0]])
    total     = len(df_ctx)
    percentil = round((1 - posicion / total) * 100)
    return f"""
    <div style="margin-top:12px; background:#0d0d14; border:1px solid #2a2a3a;
                border-radius:8px; padding:10px; text-align:center;">
        <div style="color:#aaa; font-size:10px; margin-bottom:4px;">POSICION POR PRECIO EN SU RAREZA</div>
        <div style="color:white; font-size:14px; font-weight:700;">
            #{posicion} <span style="color:#888; font-size:11px;">de {total}</span>
        </div>
        <div style="background:#1a1a25; border-radius:20px; height:8px; margin-top:8px; overflow:hidden;">
            <div style="background: linear-gradient(90deg, {color_r}, {color_r}aa);
                        width:{percentil}%; height:100%; border-radius:20px;"></div>
        </div>
        <div style="color:#888; font-size:10px; margin-top:4px;">
            Mas cara que el {percentil}% de skins de este tier
        </div>
    </div>
    """

# =====================================================================
#FUNCION PRINCIPAL DE RENDERIZADO
# =====================================================================
def renderizar(skin_nombre, rareza, anio_min, anio_max):

    # Filtro del contexto (rareza + rango de anios)
    df_ctx = df_clean[
        (df_clean["Rarity"] == rareza) &
        (df_clean["Year"] >= anio_min) &
        (df_clean["Year"] <= anio_max)
    ].copy()

    # Fila de la skin seleccionada
    fila_df = df_ctx[df_ctx["Skin Name"] == skin_nombre]
    if fila_df.empty:
        return
    fila = fila_df.iloc[0]

    precio      = fila["Precio_Clean"]
    url_img     = fila["URL_Imagen"]
    coleccion   = fila.get("Collection", "Desconocida")
    anio_lanz   = int(fila["Year"])
    ventas_skin = int(fila["Ventas_Clean"])
    color_r     = color_rareza(rareza)

    # ------------------------------------------------------------------
    # TARJETA DE LA SKIN SELECCIONADA
    # ------------------------------------------------------------------
    with zona_tarjeta:
        clear_output(wait=True)
        html = f"""
        <div style="
            background: linear-gradient(145deg, #1a1a22, #12121a);
            border: 2px solid {color_r};
            border-radius: 12px;
            padding: 18px 16px;
            font-family: 'Segoe UI', sans-serif;
            color: white;
            box-shadow: 0 0 18px {color_r}55;
            min-width: 280px;
        ">
            <div style="text-align:center; margin-bottom:8px;">
                <span style="
                    background:{color_r}22; border:1px solid {color_r};
                    color:{color_r}; font-size:10px; font-weight:700;
                    letter-spacing:2px; padding:3px 10px; border-radius:20px;
                ">{rareza.upper()}</span>
            </div>

            <h3 style="text-align:center; font-size:14px; margin:8px 0 4px 0;
                       color:#ffffff; line-height:1.4;">
                {skin_nombre}
            </h3>

            <p style="text-align:center; color:#888; font-size:11px; margin:0 0 14px 0;">
                {coleccion} &nbsp;·&nbsp; {anio_lanz}
            </p>

            <div style="
                background:#0d0d14; border:1px solid #2a2a3a; border-radius:8px;
                padding:10px; min-height:160px; display:flex;
                align-items:center; justify-content:center; margin-bottom:14px;
            ">
                <img
                    src="{url_img}"
                    alt="{skin_nombre}"
                    style="max-width:100%; max-height:150px; object-fit:contain;"
                    onerror="this.onerror=null;this.src='{URL_PLACEHOLDER}';"
                >
            </div>

            <div style="display:flex; gap:10px; justify-content:center;">
                <div style="flex:1; background:#0d0d14; border:1px solid #2a2a3a;
                            border-radius:8px; padding:10px; text-align:center;">
                    <div style="color:#55efc4; font-size:18px; font-weight:700;">{precio:,.2f}€</div>
                    <div style="color:#666; font-size:10px; margin-top:2px;">PRECIO MERCADO</div>
                </div>
                <div style="flex:1; background:#0d0d14; border:1px solid #2a2a3a;
                            border-radius:8px; padding:10px; text-align:center;">
                    <div style="color:{color_r}; font-size:18px; font-weight:700;">{ventas_skin:,}</div>
                    <div style="color:#666; font-size:10px; margin-top:2px;">VENTAS</div>
                </div>
            </div>

            {_rank_html(df_ctx, skin_nombre, precio, color_r)}
        </div>
        """
        display(HTML(html))

    # ------------------------------------------------------------------
    # GRAFICOS
    # ------------------------------------------------------------------
    with zona_graficos:
        clear_output(wait=True)

        # ---- GRAFICO 1: TOP 15 SKINS MAS CARAS DE LA RAREZA ----
        # Muestra donde se posiciona tu skin frente a las mas valiosas
        # del mismo tier. La skin aparece destacada con el color de rareza.
        top15 = (
            df_ctx.nlargest(15, "Precio_Clean")[["Skin Name", "Precio_Clean"]]
            .sort_values("Precio_Clean")
        )
        colores_barras = [color_r if n == skin_nombre else "#2d3561" for n in top15["Skin Name"]]

        fig1 = go.Figure(go.Bar(
            x=top15["Precio_Clean"],
            y=top15["Skin Name"],
            orientation="h",
            marker_color=colores_barras,
            hovertemplate="<b>%{y}</b><br>Precio: %{x:.2f}€<extra></extra>",
        ))
        fig1.update_layout(
            title=dict(
                text=(
                    f"<b>TOP 15 SKINS MAS CARAS — {rareza.upper()}</b><br>"
                    f"<sup style='color:#aaaaaa'>La barra destacada eres tu. "
                    f"Compara tu precio con los lideres del tier.</sup>"
                ),
                font=dict(size=15, color="#ffffff"),
            ),
            template="plotly_dark",
            height=420,
            margin=dict(l=10, r=20, t=80, b=30),
            xaxis=dict(title="Precio (€)", gridcolor="#2a2a3a"),
            yaxis=dict(title="", tickfont=dict(size=10)),
            plot_bgcolor="#0d0d14",
            paper_bgcolor="#12121a",
        )
        fig1.show()

        # ---- GRAFICO 2: PRECIO vs. VOLUMEN DE VENTAS ----
        # Revela si las skins caras se venden poco (exclusividad) o si hay
        # skins de precio medio con alta demanda. El color indica el año
        # de lanzamiento. La estrella naranja marca la skin seleccionada.Si le das click a la estrella de 
        # la derecha del gráfico esta se oculta
        fig2 = px.scatter(
            df_ctx,
            x="Precio_Clean",
            y="Ventas_Clean",
            hover_name="Skin Name",
            color="Year",
            color_continuous_scale="Plasma",
            size="Precio_Clean",
            size_max=18,
            opacity=0.6,
            template="plotly_dark",
            labels={
                "Precio_Clean": "Precio de Mercado (€)",
                "Ventas_Clean": "Volumen de Ventas",
                "Year": "Año de lanzamiento",
            },
            title=(
                f"<b>PRECIO VS. VOLUMEN DE VENTAS — {rareza.upper()}</b><br>"
                f"<sup style='color:#aaaaaa'>Cada punto es una skin. El color indica el anio de lanzamiento. "
                f"La estrella naranja eres tu. Para ocultar la estrella darle click <br> a la estrella de la derecha</sup>"
            ),
        )
        # Nombre corto para la etiqueta flotante (solo la parte tras "|")
        label_corto = skin_nombre.split("|")[-1].strip() if "|" in skin_nombre else skin_nombre
 
        fig2.add_trace(go.Scatter(
            x=[precio],
            y=[ventas_skin],
            mode="markers+text",
            marker=dict(symbol="star", size=22, color="#ffa502", line=dict(color="white", width=1.5)),
            text=[label_corto],
            textposition="top center",
            textfont=dict(size=9, color="#ffa502"),
            name=label_corto,       
            showlegend=True,          
            legendgroup="skin_activa",
            hovertemplate=(
                f"<b>{skin_nombre}</b><br>"
                f"Precio: {precio:.2f}€<br>"
                f"Ventas: {ventas_skin:,}<extra></extra>"
            ),
        ))
        fig2.update_layout(
            title_font=dict(size=15, color="#ffffff"),
            height=400,
            margin=dict(l=10, r=120, t=80, b=30),  
            plot_bgcolor="#0d0d14",
            paper_bgcolor="#12121a",
            # Colorbar a la izquierda del area del grafico
            coloraxis_colorbar=dict(
                title="Año",
                thickness=12,
                len=0.95,
                x=1.0,          
                xanchor="left",
                y=0.5,
                yanchor="middle",
                tickmode="linear",  
                tickformat="d"      
             
            ),
            # Leyenda discreta  en la esquina superior derecha
            legend=dict(
                x=1.02,
                y=0.95,
                xanchor="left",
                yanchor="bottom",
                bgcolor="rgba(18,18,26,0.8)",
                bordercolor="#2a2a3a",
                borderwidth=1,
                font=dict(size=10, color="#ffffff"),
            ),
            showlegend=True,
        )
        fig2.show()

        # ---- GRAFICO 3: SKINS LANZADAS POR AÑO ----
        # Muestra en que años Valve lanzo mas skins de esta rareza.
        # La barra del año de la skin aparece destacada con el color de rareza.
        df_por_anio = df_ctx.groupby("Year").size().reset_index(name="Cantidad")
        colores_anio = [color_r if int(y) == anio_lanz else "#2d3561" for y in df_por_anio["Year"]]

        fig3 = go.Figure(go.Bar(
            x=df_por_anio["Year"],
            y=df_por_anio["Cantidad"],
            marker_color=colores_anio,
            hovertemplate="Año %{x}<br>Skins lanzadas: %{y}<extra></extra>",
        ))
        fig3.update_layout(
            title=dict(
                text=(
                    f"<b>SKINS LANZADAS POR AÑO — {rareza.upper()}</b><br>"
                    f"<sup style='color:#aaaaaa'>La barra destacada corresponde al año de lanzamiento "
                    f"de tu skin seleccionada ({anio_lanz}).</sup>"
                ),
                font=dict(size=15, color="#ffffff"),
            ),
            template="plotly_dark",
            height=320,
            margin=dict(l=10, r=20, t=80, b=30),
            xaxis=dict(title="Anio", dtick=1, gridcolor="#2a2a3a"),
            yaxis=dict(title="Numero de Skins Lanzadas", gridcolor="#2a2a3a"),
            plot_bgcolor="#0d0d14",
            paper_bgcolor="#12121a",
        )
        fig3.show()

        # ---- GRAFICO 4: TOP 10 COLECCIONES POR PRECIO PROMEDIO ----
        # Indica si la coleccion de la skin es premium o de bajo valor
        # en comparacion con las 10 colecciones mas caras de la rareza.
        if "Collection" in df_ctx.columns:
            df_colec = (
                df_ctx.groupby("Collection")
                .agg(Precio_Promedio=("Precio_Clean", "mean"), Num_Skins=("Skin Name", "count"))
                .reset_index()
                .sort_values("Precio_Promedio", ascending=False)
                .head(10)
                .sort_values("Precio_Promedio")
            )
            colores_colec = [color_r if c == coleccion else "#2d3561" for c in df_colec["Collection"]]

            fig4 = go.Figure(go.Bar(
                x=df_colec["Precio_Promedio"],
                y=df_colec["Collection"],
                orientation="h",
                marker_color=colores_colec,
                customdata=df_colec["Num_Skins"],
                hovertemplate=(
                    "<b>%{y}</b><br>"
                    "Precio promedio: %{x:.2f}€<br>"
                    "Skins en esta coleccion: %{customdata}<extra></extra>"
                ),
            ))
            fig4.update_layout(
                title=dict(
                    text=(
                        f"<b>TOP 10 COLECCIONES POR PRECIO PROMEDIO — {rareza.upper()}</b><br>"
                        f"<sup style='color:#aaaaaa'>La barra destacada es la coleccion de tu skin "
                        f"({coleccion}). Precio promedio alto = coleccion premium.</sup>"
                    ),
                    font=dict(size=15, color="#ffffff"),
                ),
                template="plotly_dark",
                height=380,
                margin=dict(l=10, r=20, t=90, b=30),
                xaxis=dict(title="Precio Promedio (€)", gridcolor="#2a2a3a"),
                yaxis=dict(title="", tickfont=dict(size=10)),
                plot_bgcolor="#0d0d14",
                paper_bgcolor="#12121a",
            )
            fig4.show()

# =====================================================================
# CALLBACKS DE LOS WIDGETS
# =====================================================================
def on_cambio_rareza_o_slider(change):
    # Desconectar ANTES de modificar options/value para evitar disparos espurios
    try:
        dropdown_skin.unobserve(on_cambio_skin, names="value")
    except ValueError:
        pass

    rareza = dropdown_rareza.value
    anio_min, anio_max = slider_anios.value

    df_f = df_clean[
        (df_clean["Rarity"] == rareza) &
        (df_clean["Year"] >= anio_min) &
        (df_clean["Year"] <= anio_max)
    ]
    opciones = sorted(df_f["Skin Name"].unique().tolist()) if len(df_f) > 0 else ["(sin resultados)"]

    # Asignar opciones y valor con el observer desconectado
    dropdown_skin.options = opciones
    dropdown_skin.value   = opciones[0]

    # Reconectar y renderizar una unica vez con los valores correctos
    dropdown_skin.observe(on_cambio_skin, names="value")
    _renderizar_actual()

def on_cambio_skin(change):
    _renderizar_actual()

def _renderizar_actual():
    rareza = dropdown_rareza.value
    skin   = dropdown_skin.value
    a_min, a_max = slider_anios.value
    if skin and skin != "(sin resultados)":
        renderizar(skin, rareza, a_min, a_max)

dropdown_rareza.observe(on_cambio_rareza_o_slider, names="value")
slider_anios.observe(on_cambio_rareza_o_slider, names="value")
dropdown_skin.observe(on_cambio_skin, names="value")

# =====================================================================
# LAYOUT FINAL Y PRIMER RENDER
# =====================================================================
on_cambio_rareza_o_slider(None)

cabecera = widgets.HTML("""
<div style="
    background: linear-gradient(90deg, #12121a, #1e1e2e);
    border-bottom: 2px solid #eb4b4b;
    padding: 14px 20px;
    font-family: 'Segoe UI', sans-serif;
    display: flex;
    align-items: center;
    gap: 12px;
    margin-bottom: 4px;
    border-radius: 8px 8px 0 0;
">
    <div>
        <div style="color:white; font-size:17px; font-weight:700; letter-spacing:1px;">
            CS2 / CSGO Skin Market Dashboard
        </div>
        <div style="color:#666; font-size:11px;">
            Filtra por rareza y año · Selecciona una skin para analizar su posicion en el mercado
        </div>
    </div>
</div>
""")

controles = widgets.HBox(
    [dropdown_rareza, dropdown_skin, slider_anios],
    layout=widgets.Layout(
        padding="12px 16px",
        gap="20px",
        align_items="center",
        flex_wrap="wrap",
        border="1px solid #2a2a3a",
        margin_bottom="12px",
    ),
)

cuerpo = widgets.HBox(
    [zona_graficos, zona_tarjeta],
    layout=widgets.Layout(gap="16px", align_items="flex-start"),
)

display(widgets.VBox([cabecera, controles, cuerpo]))

Sincronizando base de datos de imagenes con la API de Valve...
La API devolvio status 404. Probando siguiente URL...
Conectado a la API: https://raw.githubusercontent.com/ByMykel/CSGO-API/main/public/api/en/skins.json
1940 skins indexadas.
Procesando dataset local...
Dataset listo. 1418 registros procesados.
